# Open-book test
A small open model answers questions about a made-up rulebook it has never seen: with no rulebook, with the whole rulebook, with 2 pages found by matching words, and with 2 pages found by meaning.

Runs on a free Colab GPU (Runtime → Change runtime type → T4 GPU), or slowly on a CPU. Answers can differ a little from run to run on different hardware.

In [1]:
# Install the library that downloads and runs open models (Colab already has numpy and torch).
!pip -q install transformers accelerate

In [2]:
# Load the two models this test needs. The first run downloads them (about 3 GB), so it takes a few minutes.
import re, numpy as np, torch
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModel

# The model that writes the answers: a small open chat model (1.5 billion numbers inside).
# It was trained before this rulebook existed, so it cannot know these rules by heart.
MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
# A much smaller model that writes nothing. It turns a piece of text into a list of 384 numbers
# that describes what the text is about, so two texts that mean the same thing get similar lists
# even when they use different words. We use it to find pages "by meaning".
EMBED = "sentence-transformers/all-MiniLM-L6-v2"

# Use the graphics card (GPU) when there is one; it is many times faster than the processor (CPU).
dev = "cuda" if torch.cuda.is_available() else "cpu"

# A model reads numbers, not letters. The tokenizer cuts text into small pieces (words and parts
# of words) and gives each piece its number; it also turns the model's numbers back into text.
tok = AutoTokenizer.from_pretrained(MODEL)
# float16 stores each of the model's numbers in half the memory, which fits a free Colab GPU.
# A CPU is slow with that format, so on a CPU we keep full size (float32).
# .eval() switches the model from training mode to plain use.
model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float16 if dev == "cuda" else torch.float32).to(dev).eval()
# The meaning model and its own tokenizer. It is small enough to stay on the CPU.
etok, emodel = AutoTokenizer.from_pretrained(EMBED), AutoModel.from_pretrained(EMBED).eval()

In [3]:
# The made-up rulebook: 28 short pages, each starting with "## " and its number.
# It was invented, so the model cannot have seen it in training: anything right it says must come from the pages we give it.
RULEBOOK = """
# Student Halls — House Rules

(A made-up university hall of residence. No real university or company.)

## 1. The hall office
The hall office is in room G04 on the ground floor. It is open on Tuesdays and Thursdays from 4 pm to 7 pm. Outside those hours, write to the office by email and expect a reply within two working days.

## 2. Paying rent
Rent is due on the 3rd day of each month and is paid by bank transfer. If the money has not arrived by the 10th, a late fee of 35 euros is added to the next month's rent.

## 3. Your deposit
Your deposit is returned within 21 days after you hand back all your keys. The office checks the room first, and the cost of any cleaning or damage beyond normal wear is taken from the deposit.

## 4. Moving out
To end your tenancy you must give nine weeks' written notice. Send the notice by email to the hall office; a phone call does not count as notice.

## 5. Quiet hours
Quiet hours are from 10:30 pm to 7 am on weeknights, and from 11:30 pm to 9 am on Friday and Saturday nights. During quiet hours, keep music, televisions and voices at a level that cannot be heard in the next room.

## 6. Animals
Each room may keep one cat, or one dog that weighs less than 8 kg when fully grown. You need written permission from the hall office before the animal moves in. Birds and other animals are not allowed.

## 7. Bicycles
Bicycles are kept in basement room B2. The door code is given out by the hall office. Bicycles may not be left in hallways, on stairs or in bedrooms.

## 8. Rubbish and recycling
The bins are behind door C in the courtyard. Glass is collected on Tuesdays and paper on Fridays. General rubbish is collected every Monday and Thursday morning.

## 9. Laundry room
The shared washing machines and dryers are on the fourth floor and can be used from 7 am to 9 pm. Machines run on tokens from the hall office; one wash costs two tokens and one dry costs one token.

## 10. Overnight guests
Guests may stay overnight for up to 14 nights in a row. A guest who stays longer must be registered with the hall office.

## 11. Parking
The building has 12 parking spaces, given out from a waiting list kept by the hall office. The two visitor spaces by the gate may be used for up to 3 hours at a time.

## 12. Heating
The central heating is switched on from 15 October to 15 April. Each radiator has its own dial; please do not cover radiators with furniture or curtains.

## 13. Repairs
Report repairs with the form on the office door or by email. The office replies within three working days. For an emergency such as a water leak, a burst pipe or no power, call the 24-hour repair line printed on the notice board in the entrance hall.

## 14. Smoking
Smoking is not allowed anywhere inside the building, including bedrooms, stairwells and the lift. Residents may smoke in the corner of the courtyard by the oak tree.

## 15. Barbecues
Charcoal and gas barbecues are not allowed anywhere in the hall or its garden. Small electric grills are allowed in the shared kitchens.

## 16. Decorating
You may put up posters with removable putty only. Nails, screws and tape on the walls are not allowed.

## 17. Subletting
You may not sublet your room without written permission from the hall office. Short holiday rentals to tourists are not allowed at all.

## 18. Keys
A lost key costs 60 euros to replace. If you are locked out outside office hours, you must call a locksmith yourself and pay the call-out fee.

## 19. Parcels
Deliveries that do not fit in your letterbox are kept in the parcel room, G02, for 10 days. After 10 days they are returned to the sender.

## 20. Moving in and moving day
Book the lift for moving through the hall office. A booking lasts up to 4 hours and is only possible on Saturdays between 9 am and 5 pm.

## 21. Common room
The common room on the ground floor can be booked up to two weeks ahead and used until 10 pm. A cleaning deposit of 50 euros is paid when you book and returned if the room is left clean.

## 22. Fire alarm
The fire alarm is tested at 10 am on the first Monday of every month. If it rings at any other time, leave the building by the stairs, not the lift.

## 23. Windows
The outsides of all windows are cleaned twice a year, in April and in October. You will get a note a week before.

## 24. Insurance
Every resident must have contents insurance for their own belongings. The building's insurance covers the building only, not your things.

## 25. Roof terrace
The roof terrace is open from May to September, between 8 am and 10 pm. Glass bottles are not allowed on the terrace.

## 26. Snow and ice
The building clears snow from the paths and the courtyard. Residents clear snow from their own window ledges.

## 27. Noise complaints
If a neighbour is too loud, first speak to them politely. If that does not help, send a written complaint to the hall office with the dates and times.

## 28. Internet in the common room
There is free Wi-Fi in the common room. The password is changed on the first day of each month and posted on the common room door.
"""

In [4]:
# The tools: split the rulebook into pages, two ways of finding the right pages, and one way of asking the model.

# Common little words that say nothing about the topic. They are ignored when we match words,
# otherwise "the" and "is" would make every page look like a match.
STOP = set("a an the of to in on for and or is are be do does i my me can what when where which how much many with at by it its this that you your will from up out after before there any all not as if".split())
# words("Can my puppy live with me?") -> {"puppy", "live"}: the lower-case words of a text, minus the little words.
words = lambda s: {w for w in re.findall(r"[a-z0-9]+", s.lower()) if w not in STOP}

def split_sections(text):
    # Cut the text into pages wherever a line starts with "## ", and number each page by the number
    # in its heading ("## 6. Animals" -> page 6). A page with no number in its heading gets its
    # position instead. Anything before the first "## " (the title) is dropped.
    out = {}
    blocks = [b.strip() for b in re.split(r"\n(?=## )", text.strip()) if b.strip().startswith("## ")]
    for i, b in enumerate(blocks):
        m = re.match(r"## (\d+)", b)
        out[int(m.group(1)) if m else i + 1] = b
    return out

def embed(texts):
    # Turn each text into its list of 384 "meaning" numbers.
    enc = etok(texts, padding=True, truncation=True, return_tensors="pt")   # text -> token numbers, padded to the same length
    with torch.no_grad():                                                     # we only use the model, so skip the bookkeeping training needs
        out = emodel(**enc).last_hidden_state                                 # 384 numbers for every token
    m = enc.attention_mask.unsqueeze(-1).float()                              # 1 for real tokens, 0 for the padding
    # Average the real tokens' numbers into one list per text, then scale the list to length 1,
    # so that later a plain multiply-and-add gives a similarity score between -1 and 1.
    return torch.nn.functional.normalize((out * m).sum(1) / m.sum(1), dim=1).numpy()

def load(text):
    # Get a document ready to search. Run it again on any text to switch documents.
    global SECTIONS, WHOLE, IDS, VEC
    SECTIONS = split_sections(text)                # {page number: page text}
    WHOLE = "\n\n".join(SECTIONS.values())          # every page in one long text, for the "whole rulebook" test
    IDS = list(SECTIONS)                           # the page numbers, in order
    VEC = embed([SECTIONS[n] for n in IDS])        # every page's meaning numbers, worked out once here

def keyword_top(q, k=2):
    # Search by matching words: a page scores one point for every word it shares with the question.
    # Highest score first; on a tie, the lower page number wins. Returns the top k page numbers.
    score = {n: len(words(q) & words(s)) for n, s in SECTIONS.items()}
    return sorted(score, key=lambda n: (-score[n], n))[:k]

def meaning_top(q, k=2):
    # Search by meaning: compare the question's meaning numbers with every page's.
    # Multiplying two lists number by number and adding it all up gives one similarity score per page
    # (close to 1 = about the same thing). Returns the k page numbers with the highest scores.
    sims = VEC @ embed([q])[0]
    return [IDS[i] for i in np.argsort(-sims)[:k]]

def ask(system, q):
    # Ask the model one question. "system" is the instruction it reads first (and, in the
    # open-book tests, the pages it may use); q is the student's question.
    msgs = [{"role": "system", "content": system}, {"role": "user", "content": q}]
    # Lay the messages out in the chat format this model was trained on, then turn them into token numbers.
    ids = tok(tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True), return_tensors="pt").input_ids.to(dev)
    with torch.no_grad():
        # Write at most 90 tokens. do_sample=False means it always takes its single most likely next
        # word instead of rolling dice, so the same question gives the same answer every time.
        out = model.generate(ids, max_new_tokens=90, do_sample=False)
    # The output starts with our own prompt; keep only the new part (the answer) and turn it back into text.
    return tok.decode(out[0][ids.shape[1]:], skip_special_tokens=True).strip()

# pages([6, 7]) -> the text of pages 6 and 7, one after the other.
pages = lambda ns: "\n\n".join(SECTIONS[n] for n in ns)
# The instruction for the closed-book test: no rulebook, only what the model learned in training.
CLOSED = "You help students living in a university hall of residence. Answer in one or two short sentences."
# The instruction for the open-book tests; {pages} is replaced by whichever pages we hand it.
BOOK = "You help students living in a university hall of residence. Answer in one or two short sentences, using the handbook pages below.\n\n{pages}"
load(RULEBOOK)

In [5]:
# The 20 test questions, each as (question, the page that answers it, a pattern that matches the right answer).
# They use everyday words the rulebook doesn't ("puppy" vs "dog", "jam jars" vs "glass"), which is what trips up the word search.
# The pattern is there for checking answers by hand; the loop below doesn't use it.
QUESTIONS = [
    ("Can my puppy live with me in my room?", 6, r"\b8\b|eight|permission"),
    ("How much warning do I have to give before I move out?", 4, r"nine|\b9\b"),
    ("When will I get my deposit back after I leave?", 3, r"\b21\b|twenty-one"),
    ("Until what time can I play music on a Saturday night?", 5, r"11:30|11\.30|23:30"),
    ("Where do I keep my bike?", 7, r"B2"),
    ("Which day do I put out my empty jam jars?", 8, r"Tuesday"),
    ("How many tokens does one wash cost?", 9, r"\btwo\b|\b2\b"),
    ("My friend wants to stay with me for three weeks. Is that OK?", 10, r"\b14\b|fourteen|regist"),
    ("When does the heating come on?", 12, r"15 October|October 15"),
    ("Can I smoke in my bedroom?", 14, r"\bnot\b|\bno\b|n't"),
    ("Can I use a charcoal barbecue in the garden?", 15, r"\bnot\b|\bno\b|n't"),
    ("I lost my key. How much will a new one cost?", 18, r"\b60\b|sixty"),
    ("How long will the building hold a delivery for me?", 19, r"\b10\b|ten"),
    ("When is the fire alarm tested?", 22, r"first Monday"),
    ("Can I rent my room out to tourists for a weekend?", 17, r"\bnot\b|\bno\b|n't"),
    ("Do I need insurance for my own belongings?", 24, r"\byes\b|must|required|need"),
    ("A pipe burst in my kitchen at midnight. What do I do?", 13, r"24-hour|24 hour|repair line"),
    ("When is the hall office open?", 1, r"Tuesday"),
    ("How long can a visitor park outside?", 11, r"\b3\b|three"),
    ("When can I use the roof terrace?", 25, r"May"),
]

# Questions the rulebook has no answer to, to see whether the model admits it or makes something up.
NOT_IN_BOOK = [
    "Is there a gym in the building?",
    "How much is the rent for a double room?",
    "Can I install a satellite dish?",
    "Are electric scooters allowed in the lift?",
    "What is the phone number of the 24-hour repair line?",
]

In [6]:
# The main test. Every question is asked four ways:
#   no rulebook    - the model answers from memory alone (it has never seen these rules, so watch it guess)
#   whole rulebook - all 28 pages are pasted in before the question
#   word search    - only the 2 pages that share the most words with the question
#   meaning search - only the 2 pages closest in meaning to the question
# The numbers in [ ] are the pages each search picked; compare them with "the right page".
# That is 80 answers, so it takes a few minutes on a GPU and much longer on a CPU.
for q, right_page, _ in QUESTIONS:
    kw, mn = keyword_top(q), meaning_top(q)
    print("Q:", q, "  | the right page:", right_page)
    print("  no rulebook   :", ask(CLOSED, q))
    print("  whole rulebook:", ask(BOOK.format(pages=WHOLE), q))
    print("  word search   ", kw, ":", ask(BOOK.format(pages=pages(kw)), q))
    print("  meaning search", mn, ":", ask(BOOK.format(pages=pages(mn)), q))
    print()

Q: Can my puppy live with me in my room?   | the right page: 6
  no rulebook   : Yes, your puppy can live with you in your room as long as it is properly trained and supervised to avoid disturbing others.
  whole rulebook: No, pets are not permitted in the hall.
  word search    [1, 3] : No, puppies are not allowed in student rooms due to health and safety regulations.
  meaning search [6, 17] : Yes, you can have a puppy if it meets the weight requirement (less than 8 kg) and has written permission from the hall office.

Q: How much warning do I have to give before I move out?   | the right page: 4
  no rulebook   : The specific amount of notice required varies by institution, but typically you should provide at least 24 hours' notice for moving out.
  whole rulebook: You have nine weeks' written notice before you can end your tenancy.
  word search    [4, 24] : You need to give nine weeks' written notice before moving out.
  meaning search [4, 24] : You need to give nine weeks' writte

In [7]:
# Questions the rulebook does not answer. The search still hands over its 2 closest pages
# (it always finds something), so watch whether the model says the pages don't cover it
# or makes up an answer that sounds right.
for q in NOT_IN_BOOK:
    mn = meaning_top(q)
    print("Q:", q, mn)
    print("  ", ask(BOOK.format(pages=pages(mn)), q), "\n")

Q: Is there a gym in the building? [1, 14]
   There is no gym in the building. 

Q: How much is the rent for a double room? [21, 2]
   I'm sorry, but I don't have information about the specific cost of a double room at this university. The handbook doesn't mention anything about room types or prices. To get accurate pricing details, it would be best to contact the university directly or check their official website. 

Q: Can I install a satellite dish? [25, 16]
   No, you cannot install a satellite dish as it is prohibited by the handbook. 

Q: Are electric scooters allowed in the lift? [22, 14]
   No, electric scooters are not allowed in the lift. 

Q: What is the phone number of the 24-hour repair line? [13, 18]
   The 24-hour repair line can be found on the notice board in the entrance hall. 



## Try your own document

In [8]:
# Swap in your own text: replace MY_TEXT and q, then run this cell.
# Pages are cut at lines that start with "## ". After this, every tool above uses your text;
# run the rulebook cell and the tools cell again to go back to the rulebook.
MY_TEXT = """## 1. First section
Paste your own text here.

## 2. Second section
Start each section with a line beginning "## ".
"""
load(MY_TEXT)                 # split your text into pages and work out their meaning numbers
q = "Your question here?"
print("pages found by meaning:", meaning_top(q))
print(ask(BOOK.format(pages=pages(meaning_top(q))), q))   # the answer, written from those pages only

pages found by meaning: [1, 2]
I'm sorry, but I cannot provide an answer to your question as there is no question provided. Please provide more information or context so that I can assist you better.
